# Building an API with FastAPI and Sakila

**Optional reference and self-study material** — The application is implemented in **`main.py`**, not in this notebook.

We have already built the same API with Flask. Now we will focus on **what changes with FastAPI**, while keeping the same Sakila database, SQL queries, and four endpoints.

**Before starting:** follow the FastAPI **`README.md`** for Conda setup (`conda activate base`), MySQL credentials in `.env`, and starting the server with `python main.py`. This notebook is a guide to the code, not a replacement for the application.

## 1. Same API, different framework

```text
Browser / requests  --GET /actors-->  FastAPI route  -->  Sakila (MySQL)
Browser / requests  <-- JSON -------- FastAPI route  <--  rows as dictionaries
```

| Method | Endpoint | Response |
| --- | --- | --- |
| GET | `/` | `Hello world!` |
| GET | `/actors` | JSON list of actors |
| GET | `/movies` | JSON list of movies |
| GET | `/search_movies?title=ACADEMY` | JSON list of matching movies |

FastAPI runs on **port 8000** in our example; Flask used **port 5000**.

## 2. Create the application and connect to Sakila

The beginning of **`main.py`** imports the libraries, loads credentials, creates the FastAPI application, and configures a logger:

```python
import logging
import os

import pymysql
import uvicorn
from dotenv import load_dotenv
from fastapi import FastAPI, HTTPException, Query
from fastapi.responses import PlainTextResponse

load_dotenv()
app = FastAPI()
logger = logging.getLogger(__name__)
```

The MySQL connection helper is the same as in Flask:

```python
def get_db_connection():
    return pymysql.connect(
        host="localhost",
        user=os.getenv("db_username"),
        password=os.getenv("db_password"),
        database="sakila",
        cursorclass=pymysql.cursors.DictCursor,
    )
```

The `.env` file stores credentials **outside the Python source code**. `DictCursor` returns rows as dictionaries. Never commit `.env` to Git.

## 3. Define routes with decorators

Our simplest endpoint does not need MySQL:

```python
@app.get("/", response_class=PlainTextResponse)
def hello():
    return "Hello world!"
```

Compare this with Flask's `@app.route("/", methods=["GET"])`.

FastAPI uses **`@app.get(...)`** to declare a GET route. `PlainTextResponse` ensures that this particular endpoint returns plain text, just like our Flask example.

After starting the server, open `http://127.0.0.1:8000/`.

## 4. Read data from MySQL and return JSON

The `/actors` route illustrates the same database workflow as in Flask:

```python
@app.get("/actors")
def get_actors():
    try:
        connection = get_db_connection()
        try:
            with connection.cursor() as cursor:
                cursor.execute(
                    "SELECT actor_id, first_name, last_name FROM actor;"
                )
                actors = cursor.fetchall()
        finally:
            connection.close()
    except pymysql.MySQLError:
        logger.exception("Database error in /actors")
        raise HTTPException(status_code=500, detail="Database request failed.")

    return actors
```

- The inner `finally` closes the connection, even if the SQL query fails.
- `logger.exception(...)` records the error and traceback in the server logs.
- `HTTPException` sends an HTTP error to the client without exposing database details.
- **No `jsonify()` is needed:** FastAPI converts the returned dictionaries into JSON automatically.

The `/movies` endpoint follows the same pattern, querying Sakila's `film` table.

## 5. Search movies with a query parameter

Consider this URL:

`http://127.0.0.1:8000/search_movies?title=ACADEMY`

- `/search_movies` is the **path**.
- `title=ACADEMY` is a **query parameter**, introduced by `?`.

FastAPI reads the parameter from the function signature:

```python
@app.get("/search_movies")
def search_movies(title: str = Query(default="")):
    title = title.strip()

    if not title:
        raise HTTPException(status_code=400, detail="Title parameter is required.")
```

Unlike Flask's `request.args.get("title", "")`, FastAPI automatically extracts `title` and uses the type annotation to document it. In our implementation, a missing or blank title returns **HTTP 400**.

The remaining SQL logic is the same as Flask:

```python
safe_title = title.replace("!", "!!").replace("%", "!%").replace("_", "!_")
pattern = f"%{safe_title}%"

query = """
    SELECT film_id, title, description, release_year
    FROM film
    WHERE title LIKE %s ESCAPE '!';
"""
cursor.execute(query, (pattern,))
```

`LIKE` performs a **partial-text search**, not fuzzy matching. `%s` is a SQL parameter placeholder; escaping `%` and `_` prevents these characters in user input from acting as SQL `LIKE` wildcards. No matches returns an empty list `[]` with HTTP **200**.

## 6. Run the server and explore the automatic documentation

Open a terminal in the folder containing `main.py`, activate the Conda environment as described in the README, and run:

```bash
python main.py
```

Uvicorn serves the application at `http://127.0.0.1:8000`. Keep the server terminal open, use a browser or a **second terminal** to send requests, and press **Ctrl+C** to stop it. `reload=True` restarts the development server after edits; do **not** use it in production.

**FastAPI generates documentation without additional route code:**

| URL | What you will see |
| --- | --- |
| `http://127.0.0.1:8000/docs` | Swagger UI: interactive API documentation |
| `http://127.0.0.1:8000/redoc` | ReDoc: alternative API documentation |
| `http://127.0.0.1:8000/openapi.json` | Machine-readable OpenAPI specification |

**Try it:** Open `/docs`, expand `GET /search_movies`, click **Try it out**, enter `ACADEMY` for `title`, and press **Execute**. Compare the result with the direct URL.

In [ ]:
import requests
import pandas as pd

response = requests.get(
    "http://127.0.0.1:8000/search_movies",
    params={"title": "ACADEMY"},
    timeout=10,
)
response.raise_for_status()

movies = response.json()
pd.DataFrame(movies).head()

The client receives JSON over HTTP and does **not** connect directly to MySQL. Compare this with the Flask notebook: the `requests` code is identical except for the port (`5000` → `8000`). If the cell raises a connection error, check that the FastAPI server is running in another terminal.

## 7. Quick checks (optional)

1. How does `@app.get("/actors")` compare with Flask's route decorator?
2. Why can FastAPI return `actors` directly instead of calling `jsonify()`?
3. Why do we close the database connection in `finally`?
4. What does `logger.exception(...)` add compared with `print()`?
5. What can you do in `/docs` that is not available by default in our simple Flask application?

## 8. Optional practice

**Exercise A — Test the existing API:** With the FastAPI server running, request `/actors` using `requests`, convert the JSON response to a DataFrame, and display the first five rows. No changes to `main.py` are needed.

In [ ]:
# Optional exercise A: complete the code.
# response = requests.get("...", timeout=10)
# response.raise_for_status()
# actors = response.json()
# pd.DataFrame(actors).head()

**Exercise B — Extend the API (optional challenge):** In a **copy** of `main.py`, add `GET /actor_count` returning the number of actors in Sakila as JSON. Reuse the same connection, `try`/`finally`, and logging pattern. Test it in `/docs` as well as your browser.

Suggested SQL: `SELECT COUNT(*) AS actor_count FROM actor;`

Keep the implementation in `main.py`, **not** in the notebook.

## 9. Flask vs FastAPI: quick comparison

| Feature | Flask | FastAPI |
| --- | --- | --- |
| Create application | `Flask(__name__)` | `FastAPI()` |
| GET endpoint | `@app.route("/actors", methods=["GET"])` | `@app.get("/actors")` |
| Query parameter | `request.args.get("title", "")` | `title: str = Query(default="")` |
| Return JSON | `jsonify(actors)` | `return actors` |
| HTTP error | `return jsonify({...}), 400` | `raise HTTPException(status_code=400, ...)` |
| Error logging | `app.logger.exception(...)` | `logger.exception(...)` |
| Server | `app.run(debug=True)` | `uvicorn.run("main:app", reload=True, ...)` |
| Auto-generated docs | Not included in our Flask example | `/docs`, `/redoc`, `/openapi.json` |

**Takeaway:** We changed the server framework, but the client still calls the same endpoints and receives the same successful JSON data. For installation, troubleshooting, and the complete code, refer to **`README.md`** and **`main.py`**.